# 01 Data Cleaning
Membersihkan file ekspor pesanan bulanan (`{Bulan}Sales{Tahun}_public.xlsx`) menjadi satu baris per `order_id`,
lalu menggabungkannya menjadi `Clean_data/master_clean_orders_FIXED.xlsx` (20.404 order, Jan 2024 – Nov 2025).

**Catatan data:** folder `RAW_PUBLIC_Dataset/` tidak disertakan di repository karena file mentah memuat data pribadi
pembeli (username, nomor resi, dan catatan pembeli). Notebook ini ditampilkan sebagai dokumentasi proses cleaning
beserta output-nya. Untuk mereproduksi analisis, mulai dari `Clean_data/master_clean_orders_FIXED.xlsx`
dan jalankan `02_analysis.ipynb`.

### import Library

In [11]:
import pandas as pd
import numpy as np
import re
import os

pd.set_option('display.max_columns', None)

## Load Data

In [12]:
import re
from datetime import datetime

def extract_bulan_label(filename):
    """
    Ekstrak label bulan (YYYY-MM) dari nama file berpola:
    '{NamaBulanInggris}Sales{Tahun}_public.xlsx'
    Contoh: 'JanuarySales2024_public.xlsx' -> '2024-01'
    """
    match = re.search(r'([A-Za-z]+)Sales(\d{4})', filename)
    if not match:
        return None
    nama_bulan, tahun = match.groups()
    try:
        bulan_num = datetime.strptime(nama_bulan, '%B').month
        return f"{tahun}-{bulan_num:02d}"
    except ValueError:
        return None

In [13]:
RAW_FOLDER = "RAW_PUBLIC_Dataset"
FILE_NAME = "NovemberSales2025_public.xlsx"
BULAN_LABEL = "2025-11"  # jan lupa diganti man  !!!!

FILE_PATH = os.path.join(RAW_FOLDER, FILE_NAME)


label_otomatis = extract_bulan_label(FILE_NAME)

if label_otomatis is None:
    raise ValueError(
        f"Nama file '{FILE_NAME}' tidak sesuai pola yang dikenali. "
        f"Cek manual sebelum lanjut -- jangan asumsikan BULAN_LABEL benar."
    )

if label_otomatis != BULAN_LABEL:
    raise ValueError(
        f"BULAN_LABEL ('{BULAN_LABEL}') tidak cocok dengan nama file '{FILE_NAME}' "
        f"(seharusnya '{label_otomatis}'). Kemungkinan kamu lupa update BULAN_LABEL "
        f"saat ganti file. Perbaiki dulu sebelum lanjut."
    )

print(f"Validasi OK: BULAN_LABEL '{BULAN_LABEL}' cocok dengan file '{FILE_NAME}'")

df_raw = pd.read_excel(FILE_PATH)
print("Shape:", df_raw.shape)

Validasi OK: BULAN_LABEL '2025-11' cocok dengan file 'NovemberSales2025_public.xlsx'
Shape: (1337, 17)


## Inspeksi Awal (Data quality check)
prosesi pengecekan data mentah

In [14]:
print("== Info tipe Data ===")
print(df_raw.dtypes)
print()
print("== Missing Values ==")
print(df_raw.isnull().sum())
print()
print("== Kolom yang ada (Untuk dibandingkan antar bulan)")
print(list(df_raw.columns))

# Validasi kolom wajib -- KECUALI kolom tanggal, itu ditangani terpisah di bawah
KOLOM_WAJIB = ['order_id','product_category','Status Pesanan','Alasan Pembatalan',
               'Opsi Pengiriman','Metode Pembayaran','Jumlah',
               'Returned quantity','Total Berat','Total Pembayaran','Kota/Kabupaten','Provinsi']
                
kolom_hilang = [k for k in KOLOM_WAJIB if k not in df_raw.columns]

if kolom_hilang:
    raise ValueError(f"Kolom wajib hilang di file ini {kolom_hilang}. lakukan struktur cek")

# Kolom tanggal ditangani terpisah, beberapa bulan pakai 'Waktu Pengiriman Diatur'
KANDIDAT_KOLOM_WAKTU = ['Waktu Pesanan Dibuat', 'Waktu Pengiriman Diatur']
kolom_waktu_dipakai = next((k for k in KANDIDAT_KOLOM_WAKTU if k in df_raw.columns), None)

if kolom_waktu_dipakai is None:
    raise ValueError("Tidak ada kolom waktu yang dikenali di file ini.")
elif kolom_waktu_dipakai != 'Waktu Pesanan Dibuat':
    print(f"PERHATIAN: 'Waktu Pesanan Dibuat' tidak ada. Memakai '{kolom_waktu_dipakai}' sebagai pengganti. "
          f"Note: Bukan waktu transaksi asli, melainkan jadwal pengiriman. "
          f"Aman digunakan untuk penanda bulanan, tetapi tidak untuk analisis granular.")
else:
    print(f"Validasi OK: '{kolom_waktu_dipakai}' tersedia.")

== Info tipe Data ===
order_id                               object
product_category                       object
Status Pesanan                         object
Alasan Pembatalan                      object
Opsi Pengiriman                        object
Waktu Pesanan Dibuat                   object
Metode Pembayaran                      object
Jumlah                                  int64
Returned quantity                       int64
Total Diskon                          float64
Total Berat                            object
Ongkos Kirim Dibayar oleh Pembeli     float64
Estimasi Potongan Biaya Pengiriman    float64
Total Pembayaran                       object
Perkiraan Ongkos Kirim                float64
Kota/Kabupaten                         object
Provinsi                               object
dtype: object

== Missing Values ==
order_id                                 0
product_category                         0
Status Pesanan                           0
Alasan Pembatalan              

In [15]:
print (df_raw['Status Pesanan'].value_counts())
print()

Status Pesanan
Selesai                                                                                   913
Batal                                                                                     121
Sedang Dikirim                                                                             74
Pesanan diterima, namun Pembeli masih dapat mengajukan pengembalian hingga 2025-12-08.     43
Pesanan diterima, namun Pembeli masih dapat mengajukan pengembalian hingga 2025-12-07.     41
Pesanan diterima, namun Pembeli masih dapat mengajukan pengembalian hingga 2025-12-06.     40
Telah Dikirim                                                                              37
Pesanan diterima, namun Pembeli masih dapat mengajukan pengembalian hingga 2025-12-05.     33
Pesanan diterima, namun Pembeli masih dapat mengajukan pengembalian hingga 2025-12-09.     25
Pesanan diterima, namun Pembeli masih dapat mengajukan pengembalian hingga 2025-12-10.      8
Pesanan diterima, namun Pembeli masih dapat m

## Cleaning: Kolom Moneter (Total Pembayaran)

In [16]:
def clean_rupiah(value): #Konversi String format ke Int Rupiah
    s= str(value).strip()
    if s in ('','nan','None'):
        return np.nan
    
    s_clean = s.replace('.', '')
    try:
        return int(s_clean)
    except ValueError:
        return np.nan

df = df_raw.copy()
df ['total_pembayaran_idr']= df["Total Pembayaran"].apply(clean_rupiah)

#Sanity check perbandingan jumlah digit sebelum dan sesudah

print("contoh sebelum >>> sesudah:")
print(df[["Total Pembayaran", "total_pembayaran_idr"]].drop_duplicates().head(10))
print()
print("Ada nilai gagal parsing (NaN)?", df["total_pembayaran_idr"].isnull().sum())


contoh sebelum >>> sesudah:
   Total Pembayaran  total_pembayaran_idr
0            18.900                 18900
1            31.990                 31990
2           126.900                126900
3            17.654                 17654
4            18.800                 18800
5            52.900                 52900
9            35.500                 35500
10           23.000                 23000
11           17.499                 17499
12           15.900                 15900

Ada nilai gagal parsing (NaN)? 0


## Cleaning: Kolom Total Berat

In [17]:
def extract_weight_gram(value):
    s = str (value).strip()
    match = re.search(r'([\d.]+)', s)
    if match:
        return float(match.group(1))
    return np.nan

df ['berat_gram']= df['Total Berat'].apply(extract_weight_gram)
print(df[['Total Berat','berat_gram']].drop_duplicates().head())
print()
print("Gagal parsing:", df['berat_gram'].isnull().sum())

  Total Berat  berat_gram
0      500 gr       500.0
1      400 gr       400.0
2     6000 gr      6000.0
5     2000 gr      2000.0
9     3600 gr      3600.0

Gagal parsing: 0


## Parsing waktu

In [18]:
df['waktu_pesanan'] = pd.to_datetime(df[kolom_waktu_dipakai], errors='coerce')

print("Gagal Parsing Tanggal :", df['waktu_pesanan'].isnull().sum())
print("range tanggal:", df['waktu_pesanan'].min(), "sampai", df['waktu_pesanan'].max())

df['sumber_kolom_waktu'] = kolom_waktu_dipakai 
df['bulan_file'] = BULAN_LABEL

Gagal Parsing Tanggal : 0
range tanggal: 2025-11-01 06:08:00 sampai 2025-11-30 23:17:00


## Derivasi Kolom Analisis Pembatalan

In [21]:
def classify_cancel_source(row):
    if row['Status Pesanan'] != 'Batal':
        return 'Tidak Batal'
    alasan_lower = str(row['Alasan Pembatalan']).lower()
    if alasan_lower.startswith('dibatalkan oleh pembeli'):
        return 'Pembeli'
    elif alasan_lower.startswith('dibatalkan secara otomatis oleh sistem'):
        return 'Sistem/Penjual (Otomatis)'
    elif alasan_lower.startswith('dibatalkan oleh penjual'):
        return 'Penjual (Manual)'
    else:
        return 'Tidak Terklasifikasi'

df['is_cancelled'] = (df['Status Pesanan'] == 'Batal').astype(int)
df['cancel_source'] = df.apply(classify_cancel_source, axis=1)

print(df['cancel_source'].value_counts())

tidak_jelas = df[df['cancel_source'] == 'Tidak Terklasifikasi']
if len(tidak_jelas) > 0:
    print("Perhatian! alasan pembatalan tidak terklasifikasi")
    print(tidak_jelas['Alasan Pembatalan'].unique())
else:
    print("Semua alasan pembatalan berhasil diklasifikasikan.")

cancel_source
Tidak Batal                  1216
Pembeli                        75
Sistem/Penjual (Otomatis)      45
Penjual (Manual)                1
Name: count, dtype: int64
Semua alasan pembatalan berhasil diklasifikasikan.


## Klasifikasi Retur, Status Efektif & Flag Anomali
__Note Penting__ Logika parsing teks ini rawan berubah kalau ada variasi kalimat <br>
baru di bulan lain

In [22]:
# Perbaikan: handle Returned quantity yang kosong (NaN) 
jumlah_nan_retur = df['Returned quantity'].isnull().sum()
print(f"Returned quantity kosong (NaN): {jumlah_nan_retur} dari {len(df)} baris")
if jumlah_nan_retur == len(df):
    print("PERHATIAN: seluruh kolom retur kosong di file ini -- retur tidak akan terdeteksi sama sekali untuk bulan ini.")

df['Returned quantity'] = df['Returned quantity'].fillna(0)

# Klasifikasi retur -- HARUS dibuat sebelum status_efektif, karena status_efektif butuh return_type
def classify_return_type(row):
    jumlah = row['Jumlah']
    retur = row['Returned quantity']
    if retur == 0:
        return 'Tidak Retur'
    elif retur < jumlah:
        return 'Parsial'
    else:
        return "Penuh"

df['return_type'] = df.apply(classify_return_type, axis=1)
df['is_returned'] = (df['Returned quantity']>0).astype(int)

# Status Efektif (VERSI FINAL) -- menangani status baru "Dalam Proses" (khusus November 2025 & bulan lain serupa)
def classify_status_efektif(row):
    status = row['Status Pesanan']
    if status == 'Batal':
        return 'Batal'
    elif status in ['Sedang Dikirim', 'Telah Dikirim'] or status.startswith('Pesanan diterima, namun'):
        return 'Dalam Proses'
    elif status == 'Selesai':
        if row['return_type'] == 'Penuh':
            return 'Retur Penuh (Efektif Batal)'
        elif row['return_type'] == 'Parsial':
            return 'Selesai (Retur Parsial)'
        else:
            return 'Selesai'
    else:
        return 'Status Tidak Dikenali'

df['status_efektif'] = df.apply(classify_status_efektif, axis=1)
df['is_settled'] = (~df['status_efektif'].isin(['Dalam Proses', 'Status Tidak Dikenali'])).astype(int)
df['is_gagal_revenue'] = df['status_efektif'].isin(['Batal', 'Retur Penuh (Efektif Batal)']).astype(int)

# Flag anomali -- hanya bayar 0 yang tidak dijelaskan oleh retur penuh ATAU pembayaran dibebaskan
df['is_pembayaran_dibebaskan'] = (df['Metode Pembayaran'] == 'Pembayaran dibebaskan').astype(int)
df['flag_anomali_bayar_nol'] = (
    (df['Status Pesanan']=='Selesai') &
    (df['total_pembayaran_idr']==0) &
    (df['return_type']!='Penuh') &
    (df['Metode Pembayaran']!='Pembayaran dibebaskan')
).astype(int)

anomali = df[df['flag_anomali_bayar_nol']==1]
print(f"Anomali belum terjelaskan :{len(anomali)}")
print(df['status_efektif'].value_counts())

belum_dikenali = df[df['status_efektif'] == 'Status Tidak Dikenali']
if len(belum_dikenali) > 0:
    print("PERHATIAN: ada status yang belum dikenali:")
    print(belum_dikenali['Status Pesanan'].unique())

Returned quantity kosong (NaN): 0 dari 1337 baris
Anomali belum terjelaskan :0
status_efektif
Selesai                        904
Dalam Proses                   303
Batal                          121
Retur Penuh (Efektif Batal)      8
Selesai (Retur Parsial)          1
Name: count, dtype: int64


## Agregrasi ke Level Order
Karena kolom baru (return_type, status_efektif, dst) sifatnya sama untuk semua baris dalam 1 order (bukan per baris produk), agregasinya cukup pakai 'first' — sama seperti kolom status lain.

In [23]:
df_order = df.groupby('order_id').agg(
    bulan_file=('bulan_file', 'first'),
    waktu_pesanan=('waktu_pesanan', 'first'),
    sumber_kolom_waktu=('sumber_kolom_waktu', 'first'),
    status_pesanan=('Status Pesanan', 'first'),
    is_cancelled=('is_cancelled', 'first'),
    cancel_source=('cancel_source', 'first'),
    alasan_pembatalan=('Alasan Pembatalan', 'first'),
    return_type=('return_type', 'first'),
    is_returned=('is_returned', 'first'),
    status_efektif=('status_efektif', 'first'),
    is_settled=('is_settled', 'first'),
    is_gagal_revenue=('is_gagal_revenue', 'first'),
    is_pembayaran_dibebaskan=('is_pembayaran_dibebaskan', 'first'),
    metode_pembayaran=('Metode Pembayaran', 'first'),
    opsi_pengiriman=('Opsi Pengiriman', 'first'),
    jumlah_kategori_berbeda=('product_category', 'nunique'),
    daftar_kategori=('product_category', lambda x: ', '.join(sorted(set(x)))),
    total_qty=('Jumlah', 'sum'),
    total_pembayaran_idr=('total_pembayaran_idr', 'first'),
    kota=('Kota/Kabupaten', 'first'),
    provinsi=('Provinsi', 'first'),
    flag_anomali_bayar_nol=('flag_anomali_bayar_nol', 'first'),
).reset_index()

print("Baris mentah:", len(df), "-> Order unik:", len(df_order))
print()
print("Distribusi status_efektif level order:")
print(df_order['status_efektif'].value_counts())
print()
print("Order settled vs menggantung:")
print(df_order['is_settled'].value_counts())

Baris mentah: 1337 -> Order unik: 1131

Distribusi status_efektif level order:
status_efektif
Selesai                        772
Dalam Proses                   250
Batal                          101
Retur Penuh (Efektif Batal)      8
Name: count, dtype: int64

Order settled vs menggantung:
is_settled
1    881
0    250
Name: count, dtype: int64


## Sanity Check Akhir

In [26]:
# Cancel rate dan gagal revenue rate
cancel_rate = df_order['is_cancelled'].mean()*100
gagal_revenue_rate = df_order['is_gagal_revenue'].mean()*100
print(f"Cancel Rate (status batal saja): {cancel_rate:.2f}%")
print(f"Gagal revenue rate (Batal dan retur penuh) {gagal_revenue_rate:.2f}%")
print()

#Validasi silang semua order Batal bayar harus bayar 0
check_batal = df_order[df_order['is_cancelled']==1]['total_pembayaran_idr']
print("semua order batal bayar 0?", (check_batal==0).all(), "|kecuali:",(check_batal!=0).sum())

# Validasi silang: semua order gagal revenue (Batal retur penuh)
check_gagal = df_order[df_order['is_gagal_revenue']==1]['total_pembayaran_idr']
print("semua order gagal-revenue bayar 0?", (check_gagal==0).all(), "kecuali:", (check_gagal!=0).sum())

# Missing value 
kolom_kunci = ['waktu_pesanan', 'status_pesanan', 'status_efektif', 'total_pembayaran_idr','metode_pembayaran']
print("Missing value per kolom kunci:")
print(df_order[kolom_kunci].isnull().sum())
print()

#Cek data anomali tersisa
print("Anomali belum terjeleskan", df_order['flag_anomali_bayar_nol'].sum())
print()

#distribusi return
print(df_order['return_type'].value_counts())

Cancel Rate (status batal saja): 8.93%
Gagal revenue rate (Batal dan retur penuh) 9.64%

semua order batal bayar 0? True |kecuali: 0
semua order gagal-revenue bayar 0? True kecuali: 0
Missing value per kolom kunci:
waktu_pesanan           0
status_pesanan          0
status_efektif          0
total_pembayaran_idr    0
metode_pembayaran       0
dtype: int64

Anomali belum terjeleskan 0

return_type
Tidak Retur    1123
Penuh             8
Name: count, dtype: int64


In [25]:
print("=== Returned quantity ===")
print("dtype:", df_raw['Returned quantity'].dtype)
print("Jumlah NaN:", df_raw['Returned quantity'].isnull().sum())
print("Contoh nilai unik:", df_raw['Returned quantity'].unique()[:15])
print()
print("=== Jumlah ===")
print("dtype:", df_raw['Jumlah'].dtype)
print("Jumlah NaN:", df_raw['Jumlah'].isnull().sum())
print("Contoh nilai unik:", df_raw['Jumlah'].unique()[:15])

=== Returned quantity ===
dtype: int64
Jumlah NaN: 0
Contoh nilai unik: [0 1 2]

=== Jumlah ===
dtype: int64
Jumlah NaN: 0
Contoh nilai unik: [  1   2   3  26   7   4  10   9   5  35   6 103  12  15 121]


## Export File ke excel

In [27]:
CLEAN_FOLDER = "Clean_data"
os.makedirs(CLEAN_FOLDER, exist_ok=True)

# simpan file bulanan
output_path = os.path.join(CLEAN_FOLDER, f"clean_orders_{BULAN_LABEL}.xlsx")
df_order.to_excel(output_path, index=False)
print("Tersimpan (bulanan):", output_path)

#Update file master kumpulan bulan ke bulan
master_path = os.path.join(CLEAN_FOLDER,"master_clean_orders.xlsx")

if os.path.exists(master_path):
    df_master_lama = pd.read_excel(master_path)

    # cegah duplikat (buang data kalau sudah pernah ada>>>pencegahan duplikat bulan)
    jumlah_sebelum = len(df_master_lama)
    df_master_lama = df_master_lama[df_master_lama['bulan_file']!=BULAN_LABEL]
    jumlah_setelah_dibuang = len(df_master_lama) 

    if jumlah_sebelum != jumlah_setelah_dibuang:
        print(f"Bulan {BULAN_LABEL} sudah pernah ada di master({jumlah_sebelum-jumlah_setelah_dibuang}baris lama dibuang, diganti versi terbaru)")

    df_master_baru = pd.concat([df_master_lama, df_order],ignore_index=True)
else :
    print("file master belum ada, membuat baru")
    df_master_baru=df_order.copy()

# pengurutan sesuai bulan dan waktu
df_master_baru = df_master_baru.sort_values(['bulan_file','waktu_pesanan']).reset_index(drop=True)

df_master_baru.to_excel(master_path, index=False)
print(f"Master ter-update: {master_path}| Total order sekarang: {len(df_master_baru)}")
print(f"Bulan yang sudah tercakup {sorted(df_master_baru['bulan_file'].unique())}")

Tersimpan (bulanan): Clean_data\clean_orders_2025-11.xlsx
Master ter-update: Clean_data\master_clean_orders.xlsx| Total order sekarang: 20404
Bulan yang sudah tercakup ['2024-01', '2024-02', '2024-03', '2024-04', '2024-05', '2024-06', '2024-07', '2024-08', '2024-09', '2024-10', '2024-11', '2024-12', '2025-01', '2025-02', '2025-03', '2025-04', '2025-05', '2025-06', '2025-07', '2025-08', '2025-09', '2025-10', '2025-11']


## Audit data 
Audit file master guna pengecekan Konsistensi antar Bulan

In [ ]:
import pandas as pd

df_master_check = pd.read_excel("Clean_data/master_clean_orders.xlsx")
print ('Total Order di master: ',len(df_master_check))
print ('Bulan yang tercakup', sorted(df_master_check['bulan_file'].unique()))
print()
print(df_master_check['bulan_file'].value_counts().sort_index())
print()
print('Total anomali di semua bulan :',df_master_check['flag_anomali_bayar_nol'].sum())
print()
print(df_master_check.groupby('bulan_file')['is_cancelled'].mean()*100)

Total Order di master:  18068
Bulan yang tercakup ['2024-01', '2024-02', '2024-03', '2024-04', '2024-05', '2024-06', '2024-07', '2024-08', '2024-09', '2024-10', '2024-11', '2024-12', '2025-01', '2025-02', '2025-03', '2025-04', '2025-05', '2025-06', '2025-07', '2025-08', '2025-09']

bulan_file
2024-01     431
2024-02     389
2024-03     453
2024-04     492
2024-05     620
2024-06     697
2024-07    1270
2024-08    1293
2024-09    1123
2024-10    1217
2024-11    1037
2024-12    1214
2025-01    1150
2025-02     957
2025-03     628
2025-04     824
2025-05     819
2025-06     615
2025-07     766
2025-08     942
2025-09    1131
Name: count, dtype: int64

Total anomali di semua bulan : 3

bulan_file
2024-01    14.849188
2024-02    15.938303
2024-03    14.128035
2024-04    18.089431
2024-05    15.322581
2024-06    12.769010
2024-07    12.283465
2024-08    15.158546
2024-09    13.891362
2024-10    13.064914
2024-11    13.982642
2024-12    13.756178
2025-01    12.608696
2025-02    12.748171
2025

In [ ]:
anomali_master = df_master_check[df_master_check['flag_anomali_bayar_nol']==1]
print(anomali_master[['order_id','bulan_file','status_pesanan','status_efektif',
                       'return_type','total_pembayaran_idr','metode_pembayaran']].to_string())

         order_id bulan_file status_pesanan status_efektif  return_type  total_pembayaran_idr      metode_pembayaran
3320  ORD_0008375    2024-07        Selesai        Selesai  Tidak Retur                     0  Pembayaran dibebaskan
3767  ORD_0008822    2024-07        Selesai        Selesai  Tidak Retur                     0  Pembayaran dibebaskan
4130  ORD_0009185    2024-07        Selesai        Selesai  Tidak Retur                     0  Pembayaran dibebaskan


In [ ]:
df_juli_raw = pd.read_excel("RAW_PUBLIC_Dataset/JulySales2024_public.xlsx")
print("Jumlah kolom Juli:", len(df_juli_raw.columns))
print(list(df_juli_raw.columns))

Jumlah kolom Juli: 17
['order_id', 'product_category', 'Status Pesanan', 'Alasan Pembatalan', 'Opsi Pengiriman', 'Waktu Pesanan Dibuat', 'Metode Pembayaran', 'Jumlah', 'Returned quantity', 'Total Diskon', 'Total Berat', 'Ongkos Kirim Dibayar oleh Pembeli', 'Estimasi Potongan Biaya Pengiriman', 'Total Pembayaran', 'Perkiraan Ongkos Kirim', 'Kota/Kabupaten', 'Provinsi']


In [ ]:
print(sorted(df_master_check['bulan_file'].unique()))
print("Total bulan tercakup:", df_master_check['bulan_file'].nunique(), "dari 24 target")

['2024-01', '2024-02', '2024-03', '2024-04', '2024-05', '2024-06', '2024-07', '2024-08', '2024-09', '2024-10', '2024-11', '2024-12', '2025-01', '2025-02', '2025-03', '2025-04', '2025-05', '2025-06', '2025-07', '2025-08', '2025-09']
Total bulan tercakup: 21 dari 24 target
